# Project 2 — Concrete Compressive Strength Prediction
**Course:** DSC8307 Data Mining, Modeling and Analytics  
**Purpose:** Predict compressive strength (MPa) from mixture composition and curing age.

## 1. Problem formulation
This is a supervised regression task. The response is concrete compressive strength in MPa; predictors are cement, blast-furnace slag, fly ash, water, superplasticizer, coarse aggregate, fine aggregate and age. Performance is evaluated using MAE, RMSE and R². A mean-prediction baseline establishes the minimum reference performance.

## 2. Imports and data loading

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
RANDOM_STATE = 42
DATA_PATH = "Concrete_Data.csv"
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.describe().T)

## 3. Exploration and data preparation
The dataset contains 1,030 rows and nine numeric columns: eight mixture/age predictors and one strength target. The supplied data dictionary gives units for the ingredients (kg/m³), age (days), and response (MPa). Inspect missing values, duplicates, distributions, and correlations before modeling.

All variables are numeric. No categorical encoding is needed. The primary analysis uses the supplied measurements without arbitrary outlier deletion: extreme strengths can be valid concrete formulations. A median imputer is included in the pipeline for reproducibility even if the supplied file has no missing values. Scaling is used for linear regression pipeline consistency; tree models do not depend on feature scaling.

In [ ]:
target = df.columns[-1]
features = df.columns[:-1].tolist()
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
display(df.corr(numeric_only=True)[[target]].sort_values(target, ascending=False))
df[features].hist(figsize=(14,9), bins=25); plt.suptitle("Predictor distributions")
plt.tight_layout(); plt.show()
plt.figure(figsize=(10,7))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0, annot=False)
plt.title("Pearson correlation matrix"); plt.tight_layout(); plt.show()

## 4. Train/test split and mean baseline
Use an 80/20 random split with a fixed seed. The baseline predicts the mean strength in the training data for every test observation. MAE and RMSE measure typical and large errors respectively; R² measures variance explained relative to predicting the test mean.

In [ ]:
X = df[features].copy()
y = df[target].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
def regression_metrics(actual, predicted):
    return {"MAE (MPa)": mean_absolute_error(actual, predicted),
            "RMSE (MPa)": mean_squared_error(actual, predicted)**0.5,
            "R2": r2_score(actual, predicted)}
baseline_pred = np.repeat(y_train.mean(), len(y_test))
print("Mean baseline:", regression_metrics(y_test, baseline_pred))

## 5. Linear and nonlinear models
- **Linear regression** provides an interpretable additive linear benchmark. It may underfit because cementitious mixtures and curing-age effects can be nonlinear and interactive.
- **Random forest regression** averages many decorrelated decision trees, capturing nonlinearities and interactions.
- **Histogram gradient boosting regression** sequentially builds trees to reduce residual error and is included as a strong nonlinear alternative.

Models are compared on the same held-out observations. A five-fold cross-validation on the training set can be used for additional robustness; the untouched test set remains the final comparison.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=400, min_samples_leaf=2,
                                           n_jobs=-1, random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=300,
        learning_rate=0.06, l2_regularization=1.0, random_state=RANDOM_STATE)
}
fitted, predictions, rows = {}, {}, []
for name, model in models.items():
    pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted[name], predictions[name] = pipe, pred
    rows.append({"Model": name, **regression_metrics(y_test, pred)})
comparison = pd.DataFrame(rows).sort_values("RMSE (MPa)")
display(comparison)

In [ ]:
# Training-set cross-validation: report fold-averaged RMSE and R2
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in models.items():
    pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv,
        scoring={"rmse":"neg_root_mean_squared_error", "r2":"r2"}, n_jobs=-1)
    cv_rows.append({"Model":name, "CV RMSE mean":-scores["test_rmse"].mean(),
                    "CV RMSE sd":scores["test_rmse"].std(),
                    "CV R2 mean":scores["test_r2"].mean()})
display(pd.DataFrame(cv_rows).sort_values("CV RMSE mean"))

## 6. Residual diagnostics and weakest predictions
For the best held-out model, inspect residuals (actual minus predicted), residuals against fitted values, and actual-versus-predicted values. Large absolute residuals identify difficult cases. Grouping absolute error by age bands helps reveal whether the model is weaker for very young, intermediate, or mature concrete. These are diagnostic patterns in this sample, not universal engineering rules.

In [ ]:
best_name = comparison.iloc[0]["Model"]
best_pred = predictions[best_name]
residuals = y_test.to_numpy() - best_pred
fig, axes = plt.subplots(1,3,figsize=(16,4))
axes[0].scatter(best_pred,residuals,alpha=.6); axes[0].axhline(0,color="black",ls="--")
axes[0].set(xlabel="Predicted strength (MPa)",ylabel="Residual (MPa)",title="Residuals vs fitted")
axes[1].hist(residuals,bins=25); axes[1].set(title="Residual distribution",xlabel="Residual (MPa)")
axes[2].scatter(y_test,best_pred,alpha=.6)
lims=[min(y_test.min(),best_pred.min()),max(y_test.max(),best_pred.max())]
axes[2].plot(lims,lims,"k--"); axes[2].set(xlabel="Actual (MPa)",ylabel="Predicted (MPa)",title="Actual vs predicted")
plt.tight_layout(); plt.show()
diagnostics = X_test.copy()
diagnostics["actual_MPa"] = y_test
diagnostics["predicted_MPa"] = best_pred
diagnostics["residual_MPa"] = residuals
diagnostics["absolute_error_MPa"] = np.abs(residuals)
display(diagnostics.nlargest(10,"absolute_error_MPa"))
diagnostics["age_band"] = pd.cut(diagnostics["Age (day)"],
    bins=[0,7,28,56,365], labels=["1–7 days","8–28 days","29–56 days","57–365 days"],
    include_lowest=True)
display(diagnostics.groupby("age_band", observed=False)["absolute_error_MPa"].agg(["mean","median","count"]))

## 7. Cement and strength: association is not causation
A predictive association between cement content and compressive strength does **not** prove what will happen if cement content is changed in a real mixture. The dataset is observational: cement varies alongside water, slag, fly ash, superplasticizer, aggregate proportions and curing age. These variables interact and may confound the simple cement-strength relationship. Correlation or feature importance measures predictive association, not the isolated causal effect of an intervention.

A real-world claim about changing cement requires a controlled experimental design (or a defensible causal design), holding other mixture conditions constant or randomizing treatment levels, specifying curing and testing conditions, replicating batches, and estimating uncertainty. Engineering standards, workability, durability, cost and environmental impacts must also be considered. A model can support formulation screening but should not replace laboratory validation or qualified engineering judgment.

In [ ]:
cement_col = features[0]
print("Pearson correlation (cement, strength):", round(df[cement_col].corr(y), 3))
plt.figure(figsize=(7,5))
plt.scatter(df[cement_col], y, alpha=.35)
plt.xlabel("Cement (kg/m³)"); plt.ylabel("Compressive strength (MPa)")
plt.title("Observed cement-strength association"); plt.tight_layout(); plt.show()
if hasattr(fitted[best_name].named_steps["model"], "feature_importances_"):
    imp = pd.Series(fitted[best_name].named_steps["model"].feature_importances_,
                    index=features).sort_values()
    display(imp)
    imp.plot(kind="barh",title=f"Feature importance — {best_name}")
    plt.xlabel("Importance"); plt.tight_layout(); plt.show()

## 8. Conclusions
The mean baseline provides a weak reference, while nonlinear tree ensembles are expected to outperform the linear model because the relationship between mixture composition, age and strength is nonlinear. In the fixed 80/20 split (random state 42), histogram gradient boosting gave the strongest test results: MAE about 2.92 MPa, RMSE about 4.46 MPa and R² about 0.923, compared with linear regression MAE about 7.75 MPa, RMSE about 9.80 MPa and R² about 0.628. Random forest was also strong (MAE about 4.03 MPa, RMSE about 5.69 MPa, R² about 0.875).

The model's errors are not uniform; inspect the diagnostic tables and plots before use. The sample is relatively small and may not cover all raw materials, production processes, curing environments or strength ranges. Validate on independent batches and avoid extrapolating beyond the observed data.